In [1]:
from langchain_core.messages.tool import tool_call
import random
from langgraph.prebuilt import ToolNode, ToolRuntime
from langchain_core.messages import ToolMessage
from langgraph.graph import StateGraph, START, END, MessagesState, MessageGraph
from typing import TypedDict, Literal, Sequence, Annotated, Final, Tuple
from IPython.display import display
from langchain_deepseek import ChatDeepSeek
from dotenv import load_dotenv
from langchain.tools import tool
from loguru import logger
from langgraph.types import Send, Command, CachePolicy

from langchain.messages import HumanMessage, AIMessage, SystemMessage

from hitl_demo.src.chat_agent import model_with_tools

load_dotenv(override=True)

model = ChatDeepSeek(
    model="deepseek-v4-flash",
    # temperature=0.7,
    extra_body={
        "thinking": {
            "type": "disabled",
        }
    }
)

@tool(parse_docstring=False)
def get_weather(city: str, runtime:ToolRuntime) -> Command:
    """
    查询指定城市的天气

    Args:
        city: 城市名称
    """
    rand_int=random.randint(1,10)
    if rand_int <8:
        logger.info("网络波动，请重试")
        raise ConnectionError("网络错误")
    return f"城市 {city} 的天气是晴朗***********"

tools = [get_weather]
model_with_tool = model.bind_tools(tools=tools)

class UserContext:
    max_attempts: int
def llm_node(state:MessagesState)->MessageGraph:
    messages = state["messages"]
    res = model_with_tool(messages)
    return {
        "messages": messages,
    }

def router(state:MessagesState)-> Literal["tool_node", END]:
    if state["messages"][-1].tool_calls:
        return "tool_node"
    return END

def wrap_tool_call(request,execute):
    max_attempts = request.runtime.context.max_attemps
    tool_msg=""
    for i in range(max_attempts):
        try:
            tool_msg = execute(request)
            break
        except ConnectionError as e:
            logger.error("工具调用失败，当前调用次数：{}， 调用次数上限： {} ，异常信息：{}",i+1,max_attempts,e)
    if not tool_msg:
        tool_msg = ToolMessage(
            tool_call_id = tool_call["id"],
            content = "调用次数达到上限，调用失败。"
        )




